# 01 — Data Loading and First Look

**Package:** `fraud_nb01-07_v2` (flat S3 layout, shared verified helpers)

**Phase 1, notebook 1 of 7.** Reads the raw landing extract from S3, audits it structurally,
emits and validates the data contract, constructs the target from the three label sources,
stamps the chronological split, and saves the payment-grain stage file.

**What this notebook does NOT do.** No value cleaning (notebook 02), no imputation or outlier
treatment (03), no dimension joins (02, after the dimensions are cleaned). Everything here is
structural: what rows exist, which are duplicates, which keys resolve, what the label is, and
which rows are train.

**Locked decisions applied here**

| Decision | Setting |
|---|---|
| Target positives | `confirmed_fraud_flag='Y'` ∪ `audit_verdict='Fraud'` ∪ chargeback `Issuer Won` |
| Target negatives | audited-legitimate ∪ released alerts |
| Unadjudicated rows | negative at `sample_weight = 0.35` |
| Split | chronological cut at 2025-07-01, stamped once as `__split` |
| Quarantined | `payment_risk_score`, `alerted_flag`, `payment_status`, `failure_reason` |

**Leakage rule.** `__split` is stamped here and never recomputed. Every downstream `.fit()`
keys off `__split == 'train'`. Drop `__split` before fitting a model and exclude it from the
drift reference — it exists in training frames and never in an inference payload.

## 0. Colab bootstrap

Keys come from **Colab Secrets** (the key icon in the left sidebar), not from a cell. Add two
secrets and enable notebook access for both:

| Secret name | Value |
|---|---|
| `AWS_ACCESS_KEY_ID` | your IAM access key id |
| `AWS_SECRET_ACCESS_KEY` | your IAM secret |

They are loaded into environment variables so that **boto3 still resolves through the default
credential chain** — the client construction below is byte-identical to what runs in production
under IRSA. Passing keys directly into `boto3.client()` would disable IRSA later and is never
done here.

A hosted runtime is a credential-leak surface: anything pasted into a cell persists in the
notebook file and its autosave history. Nothing below prints a key.

In [ ]:
%pip install -q boto3==1.43.95

## 1. Configuration and S3 helpers

Credentials resolve from the default chain (`~/.aws/credentials` locally, IRSA in-cluster). No keys in this notebook, ever.

In [ ]:
# MARKER: fraud_nb01-07_v2 :: 01_Data_Loading_and_First_Look
import io, os, json, time, hashlib, platform, importlib
from datetime import datetime, timezone
import boto3
from botocore.exceptions import ClientError
import joblib
import numpy as np
import pandas as pd
from google.colab import userdata

BUCKET, REGION = "fraud-ecommerce", "ap-south-2"
SPLIT_DATE = pd.Timestamp("2025-07-01")      # stamped in notebook 01 as __split; verified here, never recomputed
CONTRACT_VERSION = "v1"
SEED = 42
MARKER = "fraud_nb01-07_v2"
for _k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    if not os.environ.get(_k):
        os.environ[_k] = userdata.get(_k)     # Colab Secrets -> process env only; never printed or saved
s3 = boto3.client("s3", region_name=REGION)
RAW, LABELS, CONTRACTS, DATA, REPORTS = "raw/", "raw/label_sources/", "contracts/", "data/", "reports/"  # flat layout
ident = boto3.client("sts", region_name=REGION).get_caller_identity()
print("account:", ident["Account"], "| arn:", ident["Arn"])
if ident["Arn"].endswith(":root"):
    print("NOTE: running as root — accepted for Phase 1; move to an IAM principal before Phase 2")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)


def _jsonable(o):
    if isinstance(o, (np.integer, np.floating, np.bool_)):
        return o.item()
    if isinstance(o, (pd.Timestamp, datetime)):
        return o.isoformat()
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"not JSON-serialisable: {type(o).__name__}")


def read_bytes_s3(key):
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()


def put_bytes_s3(body, key):
    s3.put_object(Bucket=BUCKET, Key=key, Body=body)
    print(f"saved s3://{BUCKET}/{key}  ({len(body):,} bytes)")


def key_exists(key):
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return False
        raise


def read_s3(key):
    return pd.read_parquet(io.BytesIO(read_bytes_s3(key)))


def save_s3(df, key):
    """Parquet only; the bytes are verified to round-trip columns, dtypes and categories before upload."""
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    body = buf.getvalue()
    back = pd.read_parquet(io.BytesIO(body))
    assert list(back.columns) == list(df.columns) and len(back) == len(df), f"parquet round-trip changed shape: {key}"
    bad = [c for c in df.columns if str(back[c].dtype) != str(df[c].dtype)]
    assert not bad, f"parquet round-trip changed dtypes in {key}: {bad}"
    badcat = [c for c in df.columns if str(df[c].dtype) == "category"
              and list(back[c].cat.categories) != list(df[c].cat.categories)]
    assert not badcat, f"parquet round-trip changed categories in {key}: {badcat}"
    put_bytes_s3(body, key)


def read_json_s3(key):
    return json.loads(read_bytes_s3(key))


def save_json_s3(obj, key):
    put_bytes_s3(json.dumps(obj, indent=1, default=_jsonable).encode(), key)


def save_model_s3(obj, key):
    buf = io.BytesIO()
    joblib.dump(obj, buf)
    put_bytes_s3(buf.getvalue(), key)


def load_model_s3(key):
    return joblib.load(io.BytesIO(read_bytes_s3(key)))


# names used by notebooks 01-04 (same verified implementations underneath)
def s3_read_csv(key, **kw):
    return pd.read_csv(io.BytesIO(read_bytes_s3(key)), **kw)


s3_read_parquet, s3_read_json = read_s3, read_json_s3


def s3_write_parquet(df, key):
    save_s3(df, key)
    return f"s3://{BUCKET}/{key}"


def s3_write_json(obj, key):
    save_json_s3(obj, key)
    return f"s3://{BUCKET}/{key}"


RAW_KEYS = ([f"{RAW}{t}.csv" for t in ["payments", "orders", "order_items", "account_logins", "customers",
                                         "merchants", "cards", "devices", "ip_reputation"]]
            + [f"{LABELS}{t}.csv" for t in ["fraud_events", "audit_sample", "chargebacks"]]
            + [f"{CONTRACTS}schema_v1.json"])
_absent = [k for k in RAW_KEYS if not key_exists(k)]
assert not _absent, f"missing landing objects in s3://{BUCKET}/: {_absent}"
print(f"landing objects present: {len(RAW_KEYS)} (flat layout, bucket root)")


def run_meta(notebook):
    return {"notebook": notebook, "marker": MARKER,
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "library_versions": LIB_VERSIONS, "version_drift": VERSION_DRIFT}


LIBS = ["pandas", "numpy", "pyarrow", "scipy", "statsmodels", "sklearn", "lightgbm", "xgboost", "joblib", "boto3"]
LIB_VERSIONS = {"python": platform.python_version(),
                **{m: importlib.import_module(m).__version__ for m in LIBS}}
EXPECTED = {"pandas": "2.2.3", "numpy": "2.1.3", "pyarrow": "23.0.1", "scipy": "1.16.3", "statsmodels": "0.15.0",
            "sklearn": "1.6.1", "lightgbm": "4.6.0", "xgboost": "3.4.1", "boto3": "1.43.95"}
VERSION_DRIFT = {m: {"verified": v, "found": LIB_VERSIONS[m]} for m, v in EXPECTED.items() if LIB_VERSIONS[m] != v}
if not LIB_VERSIONS["python"].startswith("3.13."):
    VERSION_DRIFT["python"] = {"verified": "3.13.x", "found": LIB_VERSIONS["python"]}
print(LIB_VERSIONS)
print("VERSION DRIFT vs the runtime verified on 2026-09-16:", VERSION_DRIFT or "none")

## 2. Load all 12 raw files

`dtype=str` and `keep_default_na=False` are deliberate. Pandas' default NA handling silently converts `NA`, `NULL`, `N/A` and `NaN` into nulls on read — which would hide the sentinel-coded nulls this extract carries and make notebook 03 look like it has nothing to do. We want to see the raw bytes and decide ourselves what a null is.

In [ ]:
FEATURE_FILES = ['payments', 'orders', 'order_items', 'account_logins', 'customers',
                 'merchants', 'cards', 'devices', 'ip_reputation']
LABEL_FILES   = ['fraud_events', 'audit_sample', 'chargebacks']

raw = {}
for name in FEATURE_FILES:
    raw[name] = s3_read_csv(f'{RAW}{name}.csv', dtype=str,
                            keep_default_na=False, na_values=[])
for name in LABEL_FILES:
    raw[name] = s3_read_csv(f'{LABELS}{name}.csv', dtype=str,
                            keep_default_na=False, na_values=[])

inventory = pd.DataFrame([{'table': k, 'rows': len(v), 'cols': v.shape[1]}
                          for k, v in raw.items()]).set_index('table')
print(inventory.to_string())
print(f"\ntotal rows loaded: {inventory.rows.sum():,}")

## 3. First look — blanks, sentinels, whitespace

Three distinct problems that all present as "missing", counted separately because they need different treatment. A blank is absent. A sentinel is a value someone chose to mean absent. Whitespace padding is a value that will silently fail every equality comparison you write.

In [ ]:
SENTINELS = {'NA', 'N/A', 'NAN', 'NULL', 'NONE', 'UNKNOWN', 'XX', '-1', '999',
             '999999', '000000', 'AS0', 'OTHER', 'UNKNOWN BANK', 'NIL', '?', 'TBD'}

rows = []
for tbl, df in raw.items():
    for c in df.columns:
        s = df[c].astype(str)
        stripped = s.str.strip()
        blank = int((stripped == '').sum())
        sent  = int(stripped.str.upper().isin(SENTINELS).sum())
        ws    = int((s != stripped).sum())
        if blank or sent or ws:
            rows.append({'table': tbl, 'column': c, 'blank': blank, 'sentinel': sent,
                         'whitespace': ws,
                         'pct_missing': round(100 * (blank + sent) / len(df), 3)})

quality = pd.DataFrame(rows).sort_values('pct_missing', ascending=False)
print(f"{len(quality)} columns carry blanks, sentinels or padding\n")
print(quality.to_string(index=False))

### Category cardinality

The true level counts are known from the data dictionary. Anything above them is spelling drift
for notebook 02 to canonicalise, not new information.

In [ ]:
EXPECTED_LEVELS = {('payments', 'payment_method'): 7, ('payments', 'payment_gateway'): 5,
                   ('orders', 'shipping_speed'): 4, ('customers', 'city'): 14,
                   ('customers', 'email_domain_class'): 4, ('cards', 'network'): 4}
for (tbl, col), expected in EXPECTED_LEVELS.items():
    observed = raw[tbl][col].nunique()
    flag = 'OK' if observed <= expected else f'DRIFT (+{observed - expected})'
    print(f"{tbl}.{col:22s} observed={observed:>3}  expected={expected:>3}   {flag}")

print('\npayment_method levels as received:')
print(raw['payments'].payment_method.value_counts().to_string())

## 4. Timestamps

Four formats live in one column: ISO, space-separated, ambiguous `DD/MM/YYYY`, and ISO with a
`+05:30` offset. Parse them **format by format**.

Do not reach for `pd.to_datetime(col, format='mixed', dayfirst=True)`. It silently mis-parses
ISO `2025-11-02` as 11 February, and it fails silently — you get dates, just wrong ones. This
cost me a false defect report while auditing this very dataset.

In [ ]:
def parse_datetime(s):
    """Parse format by format. Never dayfirst globally — it corrupts ISO dates."""
    x = s.astype(str).str.strip().str.replace(r'\+05:30$', '', regex=True)
    out = pd.to_datetime(x, format='%Y-%m-%dT%H:%M:%S', errors='coerce')
    for fmt in ('%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M', '%Y-%m-%d'):
        m = out.isna()
        if not m.any():
            break
        out[m] = pd.to_datetime(x[m], format=fmt, errors='coerce')
    return out

pay = raw['payments'].copy()
pay['payment_ts'] = parse_datetime(pay.payment_timestamp)

unparsed = int(pay.payment_ts.isna().sum())
print(f"unparsed: {unparsed}")
assert unparsed == 0, 'timestamp parsing incomplete — inspect before continuing'
print(f"range: {pay.payment_ts.min()}  ->  {pay.payment_ts.max()}")
print(f"months covered: {pay.payment_ts.dt.to_period('M').nunique()}")
print("\nmonthly volume:")
print(pay.payment_ts.dt.to_period('M').value_counts().sort_index().to_string())

## 5. Structural de-duplication

This runs **before** the split, per leakage rule 5: near-duplicate rows straddling a split leak
train rows into test.

Three distinct cases, three different answers:

| Case | Count | Action |
|---|---|---|
| Exact duplicate rows | ~1,994 | drop — double-delivered webhooks |
| Same `payment_id`, different content | ~12 rows | keep earliest, flag for review |
| Genuine retries (`PAYR*`) — same customer and amount seconds apart, distinct id | 1,400 | **keep and flag** — retry behaviour is signal |

Dropping the retries would be the easy mistake. A customer retrying a failed payment is exactly
the kind of behaviour a fraud model should see.

In [ ]:
n_start = len(pay)

pay = pay.drop_duplicates()
n_exact = n_start - len(pay)

pk_conflict = pay.payment_id.duplicated(keep=False)
n_pk = int(pk_conflict.sum())
pk_examples = pay.loc[pk_conflict, 'payment_id'].unique()[:5].tolist()
pay = pay.sort_values('payment_ts', kind='mergesort').drop_duplicates('payment_id', keep='first')

pay['is_retry'] = pay.payment_id.str.startswith('PAYR').astype(int)

print(f"{n_start:,} -> {len(pay):,}")
print(f"  exact duplicate rows dropped : {n_exact:,}")
print(f"  conflicting-PK rows seen     : {n_pk:,}  e.g. {pk_examples}")
print(f"  retries kept and flagged     : {int(pay.is_retry.sum()):,}")
assert pay.payment_id.is_unique, 'payment_id is not unique — joins would multiply rows'

order_items = raw['order_items'].drop_duplicates()
print(f"\norder_items {len(raw['order_items']):,} -> {len(order_items):,} "
      f"({len(raw['order_items']) - len(order_items):,} duplicate line items dropped)")

dup_identity = raw['customers'].customer_id.str.endswith('_DUP')
referenced = int(pay.customer_id.isin(
    raw['customers'].loc[dup_identity, 'customer_id']).sum())
print(f"customers: {int(dup_identity.sum()):,} duplicate identities, "
      f"referenced by {referenced} payments")

## 6. Referential integrity

Every foreign key on the payment spine, checked against its dimension. Sentinels and blanks are excluded from the check so a null key is not counted as a broken one.

In [ ]:
FKS = [('order_id', 'orders', 'order_id'), ('customer_id', 'customers', 'customer_id'),
       ('merchant_id', 'merchants', 'merchant_id'), ('device_id', 'devices', 'device_id'),
       ('card_token', 'cards', 'card_token'), ('ip_asn', 'ip_reputation', 'ip_asn')]

ri = []
for col, tbl, key in FKS:
    left = pay[col].astype(str).str.strip()
    resolvable = left[(left != '') & (~left.str.upper().isin(SENTINELS))]
    orphans = int((~resolvable.isin(set(raw[tbl][key]))).sum())
    ri.append({'fk': col, 'target': tbl, 'checked': len(resolvable),
               'orphans': orphans, 'pct': round(100 * orphans / len(pay), 3)})

ri_df = pd.DataFrame(ri)
print(ri_df.to_string(index=False))

no_items = len(set(raw['orders'].order_id) - set(order_items.order_id))
print(f"\norders with zero line items: {no_items:,}")
print("\nFINDING: orphan device_ids are a fingerprint-service outage window, not corruption.")
print("They must survive as nulls with a missing-indicator, not be dropped.")

## 7. Data contract

`schema_v1.json` is validated by three consumers: this notebook, the serving API, and the drift
job. One artifact, so the contract cannot silently diverge across three places.

Validation here is **column presence and identity only**. Dtype and range enforcement belongs in
notebook 02, after cleaning — asserting dtypes against a raw extract would fail by design.

In [ ]:
contract = s3_read_json(f'{CONTRACTS}schema_v1.json')
print(f"contract version: {contract['contract_version']}")
print(f"tables described: {len(contract['tables'])}")

violations = []
for tbl, spec in contract['tables'].items():
    name = tbl.replace('.csv', '')
    if name not in raw:
        continue
    expected = {c['name'] for c in spec['columns']}
    actual = set(raw[name].columns)
    for c in expected - actual:
        violations.append({'table': name, 'issue': 'missing column', 'column': c})
    for c in actual - expected:
        violations.append({'table': name, 'issue': 'undeclared column', 'column': c})

if violations:
    print('\nCONTRACT VIOLATIONS:')
    print(pd.DataFrame(violations).to_string(index=False))
    raise ValueError('contract violated — resolve before continuing')
print('\nall declared columns present; no undeclared columns. Contract satisfied.')

POST_DECISION = sorted({c['name'] for c in contract['tables']['payments.csv']['columns']
                        if c.get('post_decision')})
print(f"\npost-decision columns flagged by the contract: {POST_DECISION}")

## 8. Leakage register

Written down here so notebook 06 and 07 inherit it rather than rediscovering it.

**Post-decision** — known only after the payment was judged. Using any of these as a feature
trains a model on the answer. `payment_risk_score` and `alerted_flag` are the incumbent engine's
own output; they are the baseline to beat, not inputs.

**Collinear by construction** — exact linear functions of other columns. These make the design
matrix singular. Notebook 06 excludes them **by name with an asserted algebraic identity**, never
by a floating-point VIF threshold, because a VIF cutoff selects a different feature set on a
different library version and silently changes the model.

In [ ]:
LEAKAGE_REGISTER = {
    'post_decision': ['payment_risk_score', 'alerted_flag', 'payment_status', 'failure_reason'],
    'baseline_only': ['payment_risk_score', 'alerted_flag'],
    'collinear_by_construction': {
        'amount_usd':      'payment_amount / 83.2',
        'amount_paise':    'payment_amount * 100',
        'fee_pct':         'processing_fee / payment_amount',
        'order_value_inr': 'duplicate of orders.order_value',
        'net_payable':     'order_value - discount_amount + shipping_charge'},
    'label_table_columns_never_features': [
        'confirmed_fraud_flag', 'action_status', 'manual_review_flag', 'fraud_loss_amount',
        'audit_verdict', 'outcome', 'reason_code', 'decision_timestamp'],
}

# Verify the stated identity on evidence, so notebook 06 excludes by name rather than belief.
# It holds on uncorrupted rows only: the raw extract also contains ~63 internal test rows whose
# derived columns were computed upstream BEFORE the amount was corrected, plus negative and
# zero amounts. Those are notebook 03's problem, not evidence against the identity.
amt = pd.to_numeric(pay.payment_amount.str.replace(',', '', regex=False), errors='coerce')
usd = pd.to_numeric(pay.amount_usd, errors='coerce')
clean_rows = amt.notna() & usd.notna() & amt.between(1, 3e6)
resid = (amt[clean_rows] / 83.2 - usd[clean_rows]).abs()
matches = int((resid < 0.01).sum())
print(f"amount_usd == payment_amount/83.2 on {matches:,} of {int(clean_rows.sum()):,} "
      f"uncorrupted rows ({100*matches/clean_rows.sum():.2f}%), max residual {resid.max():.4f}")
mismatched = int(clean_rows.sum()) - matches
print(f"rows where the derived column disagrees with its source: {mismatched:,} "
      f"-> upstream computed them before correction; flag for notebook 03")
print()
for k, v in LEAKAGE_REGISTER['collinear_by_construction'].items():
    print(f"  {k:16s} = {v}")

## 9. Protected attributes and personal data (§2.11)

Decided **now**, before modelling, and recorded in the model card.

- **Not shipped in this dataset at all:** gender, age, race, religion, disability.
- **Shipped but excluded as model inputs:** none — there are no direct protected attributes.
- **Geographic proxies requiring a recoverability check:** `city`, `city_tier`, `home_pincode`,
  `shipping_pincode`, `billing_pincode`. These can encode community and income. Notebook 07
  measures how much of each is recoverable from the retained feature set and reports it.
- **Personal data present:** `ip_address`, `device_id`, `card_token`, `customer_id`. Pseudonymous
  identifiers, but re-identifying in combination.
- **Regime:** India DPDP Act 2023. Lawful basis: fraud prevention as a legitimate use.
- **Controls:** SSE-AES256 at rest, Block Public Access on, versioning on, least-privilege read.
  Retention and automated expiry are set in Phase 3.

This is a finding to be recorded, not a section to skip.

In [ ]:
PROTECTED_DECISION = {
    'direct_protected_attributes_present': [],
    'excluded_as_inputs': [],
    'geographic_proxies_requiring_check': ['city', 'city_tier', 'home_pincode',
                                           'shipping_pincode', 'billing_pincode'],
    'pseudonymous_identifiers': ['ip_address', 'device_id', 'card_token', 'customer_id',
                                 'session_id'],
    'regime': 'India DPDP Act 2023',
    'lawful_basis': 'fraud prevention (legitimate use)',
    'proxy_recoverability_check': 'notebook 07',
    'controls': ['SSE-AES256', 'BlockPublicAccess', 'versioning', 'least-privilege read'],
    'retention_policy': 'TBD — set in Phase 3',
}
for k, v in PROTECTED_DECISION.items():
    print(f"{k:38s} {v}")

## 10. Target construction

Ground truth is not in this dataset. It is observable only through three sources with different
coverage, latency and bias, and reconciling them is a design decision rather than a cast.

| Source | Coverage | Bias |
|---|---|---|
| `fraud_events` | alerted payments only | severe — inherits the incumbent engine's blind spots |
| `audit_sample` | random sample of **non-alerted** payments (no audited payment was alerted) | none *within that stratum* — population estimates combine it with alert review (notebook 07 gate) |
| `chargebacks` | fraud the engine missed, surfaced by the issuer | card mechanisms only, and includes friendly fraud |

**Correction to the agreed spec.** We agreed positives would be confirmed alerts plus audit
fraud, with chargebacks left out. Running it that way leaves ~900 payments the issuer ruled
against the merchant sitting in the training set labelled `0`. Teaching a model that known
fraud is legitimate is worse than not having the rows at all. Chargebacks are therefore
**included as positives at a reduced sample weight**, since `Issuer Won` contains genuine fraud
mixed with friendly fraud. Notebook 07 tests the alternative (weight 0 = exclude those rows)
and the decision is recorded there on evidence.

In [ ]:
fe, au, cb = (raw[t].apply(lambda s: s.str.strip())
              for t in ['fraud_events', 'audit_sample', 'chargebacks'])

conf_fraud  = set(fe.loc[fe.confirmed_fraud_flag == 'Y', 'payment_id'])
conf_legit  = set(fe.loc[fe.confirmed_fraud_flag == 'N', 'payment_id'])
audit_fraud = set(au.loc[au.audit_verdict == 'Fraud', 'payment_id'])
audit_legit = set(au.loc[au.audit_verdict == 'Legitimate', 'payment_id'])
cb_fraud    = set(cb.loc[cb.outcome == 'Issuer Won', 'payment_id'])

positives = conf_fraud | audit_fraud | cb_fraud
negatives = (conf_legit | audit_legit) - positives

pay['is_fraud'] = pay.payment_id.isin(positives).astype(int)
pay['label_source'] = np.select(
    [pay.payment_id.isin(conf_fraud | conf_legit),
     pay.payment_id.isin(audit_fraud | audit_legit),
     pay.payment_id.isin(cb_fraud)],
    ['alert_review', 'random_audit', 'chargeback'], 'unadjudicated')

WEIGHTS = {'alert_review': 1.00, 'random_audit': 1.00,
           'chargeback': 0.60, 'unadjudicated': 0.35}
pay['sample_weight'] = pay.label_source.map(WEIGHTS)

summary = pay.groupby('label_source').agg(
    rows=('payment_id', 'size'), positives=('is_fraud', 'sum'),
    weight=('sample_weight', 'first'))
summary['positive_rate_pct'] = (100 * summary.positives / summary.rows).round(3)
print(summary.to_string())
print(f"\nobserved positive rate: {100*pay.is_fraud.mean():.4f}% "
      f"({int(pay.is_fraud.sum()):,} of {len(pay):,})")

### What the observed rate is not

The observed positive rate counts only what someone adjudicated. The unbiased audit slice is
what lets us estimate the **true** prevalence without trusting the alert stream — and the gap
between the two is the selective-labels problem, stated as a number.

In [ ]:
n_total     = len(pay)
alerted     = set(fe.payment_id)
n_alerted   = int(pay.payment_id.isin(alerted).sum())
n_conf      = int(pay.payment_id.isin(conf_fraud).sum())
audit_rate  = len(audit_fraud) / len(au)
est_true    = (n_conf + audit_rate * (n_total - n_alerted)) / n_total

print(f"alerted payments          : {n_alerted:,}")
print(f"confirmed fraud in alerts : {n_conf:,}")
print(f"audit fraud rate (unbiased on non-alerted): {100*audit_rate:.3f}%")
print(f"audited payments that were also alerted: {len(set(au.payment_id) & alerted)} "
      "(0 => the audit samples the non-alerted stratum only)")
print(f"\nESTIMATED TRUE PREVALENCE : {100*est_true:.3f}%")
print(f"OBSERVED LABEL RATE       : {100*pay.is_fraud.mean():.3f}%")
print(f"\n=> roughly {est_true/max(pay.is_fraud.mean(), 1e-9):.1f}x the fraud in this data is")
print("   never adjudicated. Recall measured against the observed label is an overestimate;")
print("   the audit slice is the only honest denominator. Notebook 07 reports both.")

## 11. Chronological split

Stamped once, here. Never recomputed downstream.

Payment IDs are sequential but the rows are shuffled and uncorrelated with time, so a naive
index split is not a temporal split. Two leakage classes are live and both are handled:

- **Temporal** — the model is forward-looking, so the cut is chronological, not random.
- **Group** — a large share of customers transact on both sides of the cut. That is the
  production condition, not a defect of the split. The controls are: chronological,
  expanding-window CV inside train (notebook 07), point-in-time features, and no identifier
  columns in the feature set (notebook 06).

In [ ]:
pay['__split'] = np.where(pay.payment_ts < SPLIT_DATE, 'train', 'test')

split_summary = pay.groupby('__split').agg(
    rows=('payment_id', 'size'), positives=('is_fraud', 'sum'),
    start=('payment_ts', 'min'), end=('payment_ts', 'max'))
split_summary['positive_rate_pct'] = (100 * split_summary.positives / split_summary.rows).round(4)
print(split_summary.to_string())

tr_cust = set(pay.loc[pay.__split == 'train', 'customer_id'])
te_cust = set(pay.loc[pay.__split == 'test', 'customer_id'])
overlap = tr_cust & te_cust
print(f"\ncustomers in both splits: {len(overlap):,} "
      f"({100*len(overlap)/pay.customer_id.nunique():.1f}% of all customers)")
print("=> returning customers are the production condition; identifiers are never features (nb06),")
print("   and CV inside train is chronological (nb07).")

assert pay.loc[pay.__split == 'train', 'payment_ts'].max() < \
       pay.loc[pay.__split == 'test', 'payment_ts'].min(), 'splits overlap in time'
print("assertion passed: no temporal overlap between train and test")

### Concept drift lives in the test window

A new Account Takeover variant emerges from 2025-08: the device signature fades while
hosting/VPN infrastructure use rises. It sits inside the test period deliberately — a model
selected on pre-July data must face a relationship that has shifted, which is what Phase 3
monitoring has to catch.

In [ ]:
drift_start = pd.Timestamp('2025-08-01')
in_test = pay.__split == 'test'
print(f"test window   : {pay.loc[in_test,'payment_ts'].min().date()} -> "
      f"{pay.loc[in_test,'payment_ts'].max().date()}")
print(f"pre-drift test rows : {int((in_test & (pay.payment_ts <  drift_start)).sum()):,}")
print(f"post-drift test rows: {int((in_test & (pay.payment_ts >= drift_start)).sum()):,}")

## 12. Save stage 01

Parquet, not CSV. CSV has no type system: every stage boundary re-parses strings and silently re-coerces dtypes, which would reintroduce exactly the sentinel problems this notebook exists to surface.

In [ ]:
stage01 = pay.drop(columns=['payment_timestamp']).copy()

print(f"shape: {stage01.shape[0]:,} rows x {stage01.shape[1]} cols")
print(f"columns: {list(stage01.columns)}")

assert stage01.payment_id.is_unique
assert stage01.__split.isin(['train', 'test']).all()
assert stage01.sample_weight.between(0, 1).all()
assert stage01.payment_ts.notna().all()

print('\n' + s3_write_parquet(stage01, f'{DATA}01_raw_loaded.parquet'))

run_record = {
    **run_meta('01_Data_Loading_and_First_Look'),
    'rows_in': int(n_start), 'rows_out': int(len(stage01)),
    'exact_duplicates_dropped': int(n_exact), 'pk_conflicts': int(n_pk),
    'retries_retained': int(stage01.is_retry.sum()),
    'observed_positive_rate': float(stage01.is_fraud.mean()),
    'estimated_true_prevalence': float(est_true),
    'split_date': str(SPLIT_DATE.date()),
    'train_rows': int((stage01.__split == 'train').sum()),
    'test_rows': int((stage01.__split == 'test').sum()),
    'customers_in_both_splits': int(len(overlap)),
    'label_weights': WEIGHTS,
    'leakage_register': LEAKAGE_REGISTER,
    'protected_attribute_decision': PROTECTED_DECISION,
    'contract_version': contract['contract_version'],
}
print(s3_write_json(run_record, f'{REPORTS}01_run_record.json'))

## Findings to carry into notebook 02

1. **25 columns** carry blanks, sentinels or whitespace padding. Sentinels must be normalised
   **before** any `.map()` — a fixed map returns `NaN` for every unmapped value, and mapping
   after imputation leaves those nulls unfillable.
2. **Category drift** in `payment_method` (22 levels vs 7 true), `payment_gateway` (15 vs 5),
   `city` (27 vs 14, including legacy Bombay / Calcutta / Madras / Bangalore).
3. **Four datetime formats**, one ambiguous. `parse_datetime()` above is the reference
   implementation — reuse it, do not rewrite it.
4. **895 orphan `device_id`s** — a service outage window. Keep as null with a missing-indicator.
   `ip_country` missingness is MAR, concentrated behind hosting ASNs, so it is informative.
5. **380 orders with no line items** — basket features will be null for these, not zero.
6. **1,400 retries retained and flagged.** Do not let notebook 02 dedupe them away.
7. **Impossible values still present** and deliberately untouched here: negative and zero
   amounts, ~63 internal test transactions, 100-year account ages, `failed_attempt_count = 255`.
   Notebook 03 owns these, and must distinguish them from the ~800 genuine wholesale accounts
   with 400–900 prior orders, which must not be capped away.
8. **`__split` is stamped.** Every `.fit()` downstream keys off `__split == 'train'`.